# 03 · Độ tương đồng câu & Xếp hạng văn bản

| Phần | Bài toán | Metric |
|---|---|---|
| A | Cho 2 câu → chấm điểm giống nhau 0–1 | Spearman |
| B | Cho câu hỏi → tìm đoạn văn liên quan nhất | Recall@k, MRR |

Hai loại model:
- **Bi-encoder:** mã hoá từng câu thành vector riêng, so cosine → nhanh, dùng để tìm kiếm.
- **Cross-encoder:** đọc 2 câu cùng lúc → chính xác hơn nhưng chậm, dùng để chấm/xếp hạng lại.

Lý thuyết: xem `knowledge/03_tuong_dong_va_xep_hang.md`

In [ ]:
# !pip install -U sentence-transformers datasets rank_bm25 scipy

## 0. Import

In [ ]:
import re

import numpy as np
import torch
from datasets import load_dataset
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample, losses
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Phần A — Chấm điểm giống nhau giữa 2 câu

In [ ]:
BI_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
CROSS_MODEL = "xlm-roberta-base"

sts = load_dataset("sentence-transformers/stsb")    # cột: sentence1, sentence2, score (0 -> 1)
sts["train"][0]

### A1. Bi-encoder chưa train

In [ ]:
def bi_encoder_scores(model, sentences1, sentences2):
    vectors1 = model.encode(sentences1, normalize_embeddings=True)
    vectors2 = model.encode(sentences2, normalize_embeddings=True)
    return (vectors1 * vectors2).sum(axis=1)        # cosine (vì vector đã chuẩn hoá)


bi_model = SentenceTransformer(BI_MODEL, device=device)

valid = sts["validation"]
scores = bi_encoder_scores(bi_model, valid["sentence1"], valid["sentence2"])
print("Spearman:", spearmanr(scores, valid["score"]).correlation)

### A2. Fine-tune bi-encoder

In [ ]:
train_examples = []
for row in sts["train"]:
    train_examples.append(InputExample(texts=[row["sentence1"], row["sentence2"]], label=float(row["score"])))

train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)
loss = losses.CoSENTLoss(bi_model)

bi_model.fit(train_objectives=[(train_loader, loss)], epochs=3, warmup_steps=100)

In [ ]:
bi_scores = bi_encoder_scores(bi_model, valid["sentence1"], valid["sentence2"])
print("Spearman sau fine-tune:", spearmanr(bi_scores, valid["score"]).correlation)

### A3. Cross-encoder (thường chính xác nhất)

In [ ]:
cross_model = CrossEncoder(CROSS_MODEL, num_labels=1, device=device)

cross_loader = DataLoader(train_examples, shuffle=True, batch_size=16)
cross_model.fit(train_dataloader=cross_loader, epochs=3, warmup_steps=100)

In [ ]:
pairs = list(zip(valid["sentence1"], valid["sentence2"]))
cross_scores = cross_model.predict(pairs)
print("Spearman cross-encoder:", spearmanr(cross_scores, valid["score"]).correlation)

### A4. Kết hợp 2 model

In [ ]:
from scipy.stats import rankdata

for w in [0.5, 0.7, 0.9]:
    combined = w * rankdata(cross_scores) + (1 - w) * rankdata(bi_scores)
    print(w, spearmanr(combined, valid["score"]).correlation)

# Phần B — Tìm kiếm / Xếp hạng văn bản

Quy trình:
```
câu hỏi ──► BM25 (theo từ khoá) ──┐
        └─► Dense (theo nghĩa)  ──┴─► gộp kết quả ──► top 50 ──► cross-encoder xếp hạng lại ──► kết quả
```
Dữ liệu mẫu: câu hỏi SQuAD → tìm đoạn văn chứa câu trả lời.

In [ ]:
DENSE_MODEL = "intfloat/multilingual-e5-base"
RERANK_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

squad = load_dataset("rajpurkar/squad", split="validation").select(range(1000))

documents = list(dict.fromkeys(squad["context"]))      # các đoạn văn, bỏ trùng
doc_index = {doc: i for i, doc in enumerate(documents)}

questions = squad["question"]
correct_doc = [doc_index[c] for c in squad["context"]]   # đáp án: đoạn văn đúng của mỗi câu hỏi

print(len(documents), "đoạn văn,", len(questions), "câu hỏi")

### B1. Hàm đánh giá

In [ ]:
def evaluate(score_matrix, correct_doc):
    # score_matrix[i, j] = điểm của đoạn j cho câu hỏi i
    ranks = []
    for i, correct in enumerate(correct_doc):
        order = np.argsort(-score_matrix[i])                  # sắp xếp giảm dần
        rank = int(np.where(order == correct)[0][0]) + 1      # đoạn đúng đứng thứ mấy
        ranks.append(rank)
    ranks = np.array(ranks)

    return {
        "Recall@1": (ranks <= 1).mean(),
        "Recall@10": (ranks <= 10).mean(),
        "MRR@10": np.where(ranks <= 10, 1 / ranks, 0).mean(),
    }

### B2. BM25 — tìm theo từ khoá

In [ ]:
from rank_bm25 import BM25Okapi


def split_words(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([split_words(doc) for doc in documents])
bm25_scores = np.array([bm25.get_scores(split_words(q)) for q in questions])

print("BM25:", evaluate(bm25_scores, correct_doc))

### B3. Dense — tìm theo nghĩa
Model e5 cần thêm `query: ` và `passage: ` ở đầu.

In [ ]:
dense_model = SentenceTransformer(DENSE_MODEL, device=device)

doc_vectors = dense_model.encode(["passage: " + d for d in documents], normalize_embeddings=True)
question_vectors = dense_model.encode(["query: " + q for q in questions], normalize_embeddings=True)

dense_scores = question_vectors @ doc_vectors.T
print("Dense:", evaluate(dense_scores, correct_doc))

### B4. Gộp 2 kết quả bằng RRF
Mỗi đoạn được cộng điểm `1 / (60 + thứ hạng)` từ mỗi cách tìm.

In [ ]:
def to_ranks(score_matrix):
    return np.argsort(np.argsort(-score_matrix, axis=1), axis=1) + 1


def rrf(score_matrix_a, score_matrix_b, k=60):
    return 1 / (k + to_ranks(score_matrix_a)) + 1 / (k + to_ranks(score_matrix_b))


hybrid_scores = rrf(bm25_scores, dense_scores)
print("BM25 + Dense:", evaluate(hybrid_scores, correct_doc))

### B5. Xếp hạng lại top 50 bằng cross-encoder

In [ ]:
reranker = CrossEncoder(RERANK_MODEL, device=device)
TOP_K = 50

rerank_scores = np.full(hybrid_scores.shape, -1e9)
for i, question in enumerate(questions):
    top_docs = np.argsort(-hybrid_scores[i])[:TOP_K]
    pairs = [(question, documents[j]) for j in top_docs]
    rerank_scores[i, top_docs] = reranker.predict(pairs)

print("+ Rerank:", evaluate(rerank_scores, correct_doc))

---
## Nâng cao
- Fine-tune dense model bằng `losses.MultipleNegativesRankingLoss` với cặp (câu hỏi, đoạn đúng).
- Thêm **hard negative**: đoạn BM25 xếp cao nhưng sai → model học phân biệt tốt hơn.
- Tiếng Việt: tách từ (pyvi) trước BM25; thử `BAAI/bge-m3` (dense) và `BAAI/bge-reranker-v2-m3` (rerank).